# From Static Prompts to LangChain Chains with Groq

**Bano Qabil — Beginner Classroom Notebook**

This lesson starts with a prompt written directly in Python and gradually converts it into a reusable LangChain chain. The examples use the **Groq API** through `ChatGroq`.

### Learning goals

By the end, you will be able to:

1. Explain the difference between a system message and a user message.
2. Call a Groq-hosted chat model through LangChain.
3. Replace fixed text with variables in `ChatPromptTemplate`.
4. Build a chain with the LCEL pipe operator `|`.
5. Reuse one chain with different inputs.
6. Connect two small chains to make a simple workflow.

## Step 0 — VS Code / local Jupyter setup

In VS Code, select a Python kernel from the top-right corner, then run cells in order. Keep `.env` private and do not upload it to GitHub.

In [1]:
%pip install -qU langchain-core langchain-groq python-dotenv ipykernel


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# load_dotenv() reads private values from a local .env file.
# Create .env beside this notebook and add: GROQ_API_KEY=your_key_here
import os
from getpass import getpass
from dotenv import load_dotenv

load_dotenv()
groq_key = os.getenv("GROQ_API_KEY")

# Friendly fallback for a classroom computer without a .env file.
if not groq_key:
    groq_key = getpass("Paste your Groq API key: ")
    os.environ["GROQ_API_KEY"] = groq_key

if not groq_key:
    raise ValueError("GROQ_API_KEY was not provided.")

print("Groq API key is ready.")


Groq API key is ready.


## Step 1 — Import the building blocks

Think of our application as a small pipeline:

**Prompt → Groq model → plain text output**

`ChatPromptTemplate` prepares messages, `ChatGroq` talks to the model, and `StrOutputParser` gives us a normal Python string.

In [3]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda

# One model setting is reused throughout the notebook.
# This model ID is in Groq's catalog. ChatGroq + GROQ_API_KEY mean
# the request is sent through Groq even though the ID contains a vendor name.
# temperature=0 keeps classroom demonstrations more consistent.
MODEL_NAME = "openai/gpt-oss-20b"
llm = ChatGroq(model=MODEL_NAME, temperature=0)
parser = StrOutputParser()

print("Model connected through Groq:", MODEL_NAME)


Model connected through Groq: openai/gpt-oss-20b


## Step 2 — First static prompt

A **static prompt** has fixed text. It is useful for learning, but changing the task means editing the code.

- **System message:** sets the role, rules, tone, or boundaries.
- **User message:** contains the question or task.

In [4]:
# These messages are fixed: there are no variables yet.
static_messages = [
    ("system", "You are a friendly Generative AI teacher. Explain in very simple English and add one short Roman Urdu line."),
    ("human", "Explain an API keys  in 5 lines."),
]

response = llm.invoke(static_messages)
print(response.content)


1. An API key is a secret code that lets a computer program talk to another program.  
2. It shows the second program who you are and what you’re allowed to do.  
3. The key is usually a long string of letters and numbers.  
4. If someone steals the key, they could misuse the program, so keep it safe.  
5. Many services give you a key when you sign up, and you paste it into your code.  

API key se aap apni app ko secure karte hain.


### Observe: what does the model return?

`llm.invoke(...)` returns an AI message object. Its generated text is stored in `.content`. Later, the output parser will remove this extra step for us.

In [5]:
print("Python type:", type(response))
print("Only the text:", response.content)


Python type: <class 'langchain_core.messages.ai.AIMessage'>
Only the text: 1. An API key is a secret code that lets a computer program talk to another program.  
2. It shows the second program who you are and what you’re allowed to do.  
3. The key is usually a long string of letters and numbers.  
4. If someone steals the key, they could misuse the program, so keep it safe.  
5. Many services give you a key when you sign up, and you paste it into your code.  

API key se aap apni app ko secure karte hain.


## Step 3 — Change the instructions and compare

The topic below is different from the original repository example. Change only the system role and notice how the same user topic can be explained for a different audience.

In [6]:
comparison_messages = [
    ("system", "You are a patient teacher for a 15-year-old beginner. Use one everyday analogy and no difficult words."),
    ("human", "What is the difference between a prompt and a model?"),
]

comparison_response = llm.invoke(comparison_messages)
print(comparison_response.content)


Think of a **model** like a big recipe book that has lots of recipes for many dishes.  
A **prompt** is like the one dish you ask the chef to make from that book.

- **Model (recipe book)** – It contains all the instructions and knowledge. It knows how to cook many things, but it doesn’t cook anything until you ask it to.  
- **Prompt (dish request)** – It tells the chef exactly what you want: “Make me a chocolate cake.” The prompt is the question or instruction you give to the model.

So the model is the whole set of knowledge, and the prompt is the specific request you give it to get an answer.


## Step 4 — Why static prompts become a problem

Suppose 100 students ask about different topics. Copying and editing the whole prompt 100 times is error-prone. We need placeholders such as `{topic}`, `{audience}`, and `{language}`.

In [7]:
# A template is like a reusable form. Values will be inserted later.
teaching_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a helpful teacher for {audience}. Explain in {language}. "
     "Use simple words, one analogy, and one practical example."),
    ("human", "Teach me this topic: {topic}"),
])

print(teaching_prompt)


input_variables=['audience', 'language', 'topic'] input_types={} partial_variables={} messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['audience', 'language'], input_types={}, partial_variables={}, template='You are a helpful teacher for {audience}. Explain in {language}. Use simple words, one analogy, and one practical example.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['topic'], input_types={}, partial_variables={}, template='Teach me this topic: {topic}'), additional_kwargs={})]


## Step 5 — Fill the template without calling the model

This is an important debugging habit: inspect the final messages first. It costs no API call and helps you catch missing or incorrect variables.

In [8]:
filled_messages = teaching_prompt.invoke({
    "audience": "beginner Python students",
    "language": "simple English with key terms in Roman Urdu",
    "topic": "temperature in an LLM",
})

print(filled_messages.to_string())


System: You are a helpful teacher for beginner Python students. Explain in simple English with key terms in Roman Urdu. Use simple words, one analogy, and one practical example.
Human: Teach me this topic: temperature in an LLM


## Step 6 — Run the pieces manually

Before making a chain, run each part yourself. This makes the data flow easy to see.

In [9]:
# 1) The prompt creates messages.
prompt_value = teaching_prompt.invoke({
    "audience": "first-semester university students",
    "language": "simple English",
    "topic": "tokens in a language model",
})

# 2) The model reads those messages and returns an AI message.
ai_message = llm.invoke(prompt_value)

# 3) The parser converts the AI message into a plain string.
plain_text = parser.invoke(ai_message)

print(plain_text)


**What are “tokens” in a language model?**

A language model (like the one you’re talking to now) learns to predict words.  
But it doesn’t look at whole sentences at once; it looks at *small pieces* of text called **tokens**.

---

### 1.  What a token is

* **Token = a chunk of text**  
  It can be a word, part of a word, or even a punctuation mark.  
  The model splits the input into these chunks so it can count how often they appear and learn patterns.

* **Why split?**  
  Splitting into tokens lets the model handle very long sentences, different languages, and words it has never seen before.

---

### 2.  Analogy

Think of a language model like a **library of LEGO bricks**.

* Each **brick** is a token.  
* The model builds sentences by snapping bricks together.  
* If you want a new sentence, you pick the right bricks (tokens) and connect them in the right order.

Just as you can build many different structures from the same set of bricks, the model can generate many different s

## Step 7 — Convert the same flow into an LCEL chain

LCEL means **LangChain Expression Language**. The `|` symbol passes the output of one component into the next component. It is similar to a pipe carrying water forward.

In [10]:
# Read this line from left to right:
# input dictionary -> formatted prompt -> Groq model -> Python string
teaching_chain = teaching_prompt | llm | parser

answer = teaching_chain.invoke({
    "audience": "Bano Qabil beginners",
    "language": "simple English and Roman Urdu",
    "topic": "why LangChain is useful",
})

print(answer)


**Why LangChain is useful**

**Simple English**  
LangChain is a tool that helps you build programs that talk to large language models (like ChatGPT) in a smart way.  
It gives you ready‑made “building blocks” so you don’t have to write everything from scratch.  
With LangChain you can:

* **Chain steps together** – ask the model a question, get a short answer, then ask it to explain that answer, and so on.  
* **Store memory** – keep track of what the user has said so the conversation feels natural.  
* **Use external data** – let the model read a PDF, a database, or a website and then answer questions about it.  
* **Control prompts** – change the wording of the question or the style of the answer without touching the model code.  

Because of these features, you can create useful AI apps faster and with less code.

---

**Roman Urdu**  
LangChain aik framework hai jo aap ko language models ko istemal karne mein madad karta hai.  
Iske building blocks se aap:

* **Steps chain kar sak

## Step 8 — Reuse the same chain

The chain is now reusable. We change only the input dictionary; the pipeline stays the same.

In [11]:
class_topics = [
    "system prompt versus user prompt",
    "few-shot prompting",
    "hallucination in an LLM",
]

for topic in class_topics:
    result = teaching_chain.invoke({
        "audience": "beginner Generative AI students",
        "language": "simple English",
        "topic": topic,
    })
    print(f"\n--- {topic.upper()} ---")
    print(result)



--- SYSTEM PROMPT VERSUS USER PROMPT ---
**What are the two kinds of prompts?**

| Prompt type | Who writes it? | What it does |
|-------------|----------------|--------------|
| **System prompt** | The *developer* or *designer* of the AI. | It tells the AI how to behave for the whole conversation. Think of it as the “rules” or “personality” that stay the same while you chat. |
| **User prompt** | You, the *user*. | It is the question or request you give each time you talk to the AI. It changes from one turn to the next. |

---

### Simple analogy  
Imagine a **restaurant**.

* **System prompt** = the chef’s recipe book.  
  The chef (the AI) follows the recipe book to decide how to cook everything: what style, what ingredients, how fancy the dish should be. The recipe book doesn’t change while you’re eating.

* **User prompt** = your order.  
  Every time you sit down, you tell the chef what you want: “I’d like a salad with tomatoes.” The chef uses the recipe book (system prompt) to 

## Step 9 — Add our own Python function as a chain step

A chain is not limited to prompts and models. A normal Python function can clean, label, or save the result. `RunnableLambda` turns our function into a LangChain component.

In [12]:
def add_classroom_heading(text: str) -> str:
    """Add a consistent heading to every generated lesson."""
    return "BANO QABIL — QUICK STUDY NOTE\n" + "=" * 35 + "\n" + text.strip()

heading_step = RunnableLambda(add_classroom_heading)
formatted_chain = teaching_prompt | llm | parser | heading_step

formatted_answer = formatted_chain.invoke({
    "audience": "students who know basic Python",
    "language": "simple English",
    "topic": "prompt templates",
})

print(formatted_answer)


BANO QABIL — QUICK STUDY NOTE
## What is a Prompt Template?

A **prompt template** is a ready‑made “skeleton” of a question or instruction that you can fill in with different words or numbers.  
Think of it like a **recipe**: the recipe tells you the steps and the ingredients, but you can change the amount of salt, the type of meat, or the cooking time.  
With a prompt template you keep the structure of the question the same, but you swap in the parts that change.

---

### Why Use Prompt Templates?

| Problem | Prompt Template Solution |
|---------|--------------------------|
| You need to ask the same type of question many times. | Write the question once, then fill in the variable parts. |
| You want to keep the wording consistent. | The template guarantees the same phrasing each time. |
| You want to automate prompt creation in code. | Use Python string formatting to build the prompt on the fly. |

---

## How to Build a Prompt Template in Python

Python gives you a few ways to ins

## Step 10 — Build a simple two-stage chain

Now one chain writes a short lesson. A second prompt receives that lesson and creates a quiz.

**Topic → Lesson chain → Quiz prompt → Groq model → Quiz text**

In [13]:
lesson_prompt = ChatPromptTemplate.from_messages([
    ("system", "You teach Generative AI to beginners. Use simple English and stay under 120 words."),
    ("human", "Write a short lesson about {topic}. Include one analogy and one example."),
])

quiz_prompt = ChatPromptTemplate.from_messages([
    ("system", "You create easy classroom quizzes. Use only the lesson supplied by the user."),
    ("human",
     "LESSON:\n{lesson}\n\nCreate 3 multiple-choice questions. "
     "Give options A, B, C and write an answer key at the end."),
])

lesson_chain = lesson_prompt | llm | parser

# The dictionary sends lesson_chain's text into the {lesson} placeholder.
lesson_to_quiz_chain = {"lesson": lesson_chain} | quiz_prompt | llm | parser

quiz = lesson_to_quiz_chain.invoke({"topic": "static prompts and dynamic prompts"})
print(quiz)


**Quiz: Static vs. Dynamic Prompts**

1. **Which statement best describes a static prompt?**  
   A) It changes based on user input.  
   B) It remains fixed and never changes.  
   C) It always contains brackets for substitution.  

2. **Which example is a dynamic prompt?**  
   A) “Write a poem about the sea.”  
   B) “Write a poem about the ocean.”  
   C) “Write a poem about the *[user’s favorite animal]*.”  

3. **Which analogy best describes a static prompt?**  
   A) A recipe that always uses the same ingredients.  
   B) A menu that lets you pick a dish each time.  
   C) A dynamic menu that changes with each order.  

---

### Answer Key  
1. **B**  
2. **C**  
3. **A**


## Step 11 — Turn the chain into a tiny interactive app

This final cell accepts student input. It shows that the same chain can sit behind a website, chatbot, or mobile app later.

In [14]:
student_topic = input("Enter a Generative AI topic: " ).strip()
student_language = input("Choose language (English / Urdu / Roman Urdu): " ).strip()

if not student_topic:
    student_topic = "prompt engineering"
if not student_language:
    student_language = "simple English"

student_note = formatted_chain.invoke({
    "audience": "beginner students",
    "language": student_language,
    "topic": student_topic,
})

print("\n" + student_note)



BANO QABIL — QUICK STUDY NOTE
**What is AI?**  
AI stands for **Artificial Intelligence**.  
It is a way to make a computer or a machine think and act a little like a human brain.

---

### Simple explanation  
- **Artificial** – it is made by people, not natural.  
- **Intelligence** – it can learn, solve problems, and make decisions.

So, AI is a computer program that can learn from data, recognize patterns, and do tasks that normally need human thinking.

---

### One analogy  
Think of AI like a **smart robot helper**.  
Just as a robot can be taught to clean a room, a smart AI can be taught to read pictures, answer questions, or play a game. The robot follows instructions, but the AI learns from examples and gets better over time.

---

### One practical example  
**Voice assistants on phones (Siri, Google Assistant, Alexa)**  
- You ask a question: “What’s the weather today?”  
- The AI listens, understands the words, looks up the weather data, and speaks the answer back to you.

## Practice tasks

1. Change the system message so the model acts as an electronics teacher.
2. Add a `{length}` variable and request either 5 lines or 10 lines.
3. Make the teaching chain explain **embeddings** in Roman Urdu.
4. Change the custom Python function so it adds your own institute name.
5. Extend the two-stage chain so it produces 5 quiz questions instead of 3.

### Key idea

A prompt is one instruction. A template makes the instruction reusable. A chain connects prompting, the model, parsing, and other steps into one repeatable workflow.

### References

- [Original beginner notebook used as the structural starting point](https://github.com/techsimpluslearning/Complete-GenAI-Course-Python-Langchain/blob/main/notebooks/2_prompt_cains.ipynb)
- [Groq supported models](https://console.groq.com/docs/models)
- [LangChain ChatGroq integration](https://docs.langchain.com/oss/python/integrations/chat/groq)